Az alábbi scriptek a Petőfi Irodalmi Múzeum OPAC felületéről letöltött rekordok összefésülését végzi.
1. A script a JSON formátumra alakított, leltári számmal kiegészített XML fájlt beolvassa, és struktuált XLSX formátumban visszadja.
2. Az XLSX fájlt a további műveletek elvégzésére visszaalakítja JSON formátumba.
3. A script ellenőrzi, hogy a lekérdezett kapcsolatokban vannak-e olyan új entitások, akik a nodes (namespace) fájlban nem szerepelnek. Ha egyezést talál, a hozzá tartozó kapcsolatokat kiszűri, ezek ugyanis egy korábbi lekérdezésből már bekerültek az edges fájlba.
4. Az új entitásokat tartalmazó kapcsolatokat kimenti egy new_entites.json fájlba.
5. A kimeneti JSON fájl XLSX formátumra konvertálását követően az új kapcsolatok hozzáilleszthetők a meglévőkhöz.

In [ ]:
import pandas as pd
import json

In [ ]:
import xml.etree.ElementTree as ET
import re
import requests
from bs4 import BeautifulSoup
import time

In [ ]:
# A notebook blokk a PIM OPAC-ból származó XML rekordokat dolgozza fel. 
# Kinyeri a record.xml fájlban található rekordok hiányzó leltári számait a PIM OPAC felületéről, 
# Majd JSON fájlban exportálja a leltári számmal gazdagított rekordokat 'koztes_json' néven.
# A neve azért köztes json, mert ez a köztes lépés a frissen lekérdezett rekordok
# meglévő kapcsolatokhoz történő illesztésében, amit az ez alapján előállított 'new_entities'
# alapján lehet elvégezni. 

def extract_pim_id(url):
    if url:
        return url.strip().split('/')[-1].replace('PIM', '')
    return None

def fetch_leltari_szam_from_web(bib_id):
    """Lekéri a PIM weblapjáról a pontos raktári jelzetet a HTML struktúra alapján."""
    if not bib_id:
        return None
    url = f"https://resolver.pim.hu/bib/{bib_id}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. Módszer: Megkeressük a leltári szám specifikus sorát a class alapján
        inv_row = soup.find('div', class_='data-wrapper-inventoryNumber')
        if inv_row:
            val_cell = inv_row.find('div', class_='metadata-value')
            if val_cell:
                return val_cell.get_text(strip=True)
        
        # 2. Biztonsági tartalék: Ha a fenti osztálynév hiányozna, a megnevezés alapján keressük meg a mellette lévő értéket
        for name_cell in soup.find_all('div', class_='metadata-name'):
            if "Leltári szám" in name_cell.get_text():
                val_cell = name_cell.find_next_sibling('div', class_='metadata-value')
                if val_cell:
                    return val_cell.get_text(strip=True)
                    
        return None
    except Exception:
        return None

# --- XML Feldolgozás és Web Scraping ---
bemeneti_xml = "record.xml"
koztes_json = "opac_records.json"

with open(bemeneti_xml, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# Névterek eltávolítása a biztonságos kereséshez
xml_content = re.sub(r'\sxmlns(:[a-zA-Z0-9_-]+)?="[^"]+"', '', xml_content)
if "<collection>" not in xml_content and xml_content.count("<record") > 1:
    xml_content = re.sub(r'<\?xml[^>]+\?>', '', xml_content)
    xml_content = f"<collection>{xml_content}</collection>"
    
root = ET.fromstring(xml_content.strip())
records = [root] if root.tag == 'record' else root.findall('.//record')

extracted_data = []
print(f"Összesen {len(records)} rekord feldolgozása és adatok lekérése a webről...")

for idx, record in enumerate(records):
    data = {}
    
    # Alapadatok kinyerése az XML-ből
    cf_001 = record.find("controlfield[@tag='001']")
    bib_id = cf_001.text.strip() if cf_001 is not None and cf_001.text else None
    data['bib.rekord'] = bib_id
    
    # Alapértelmezett leltári szám az XML-ből
    df_090_a = record.find("datafield[@tag='090']/subfield[@code='a']")
    data['leltari_szam'] = df_090_a.text.strip() if df_090_a is not None and df_090_a.text else "KLE"
    
    # Terjedelem
    df_300 = record.find("datafield[@tag='300']")
    weight_parts = [sf.text.strip() for sf in (df_300.findall("subfield") if df_300 is not None else []) if sf.attrib.get('code') in ['a', 'b'] and sf.text]
    data['weight'] = " ".join(weight_parts) if weight_parts else None
    
    # Dátum és Megjegyzés
    df_260_c = record.find("datafield[@tag='260']/subfield[@code='c']")
    data['date'] = df_260_c.text.strip() if df_260_c is not None and df_260_c.text else None
    df_500_a = record.find("datafield[@tag='500']/subfield[@code='a']")
    data['megjegyzes'] = df_500_a.text.strip() if df_500_a is not None and df_500_a.text else None
    
    # Source
    df_100 = record.find("datafield[@tag='100']")
    if df_100 is not None:
        sf_1 = df_100.find("subfield[@code='1']")
        data['source'] = extract_pim_id(sf_1.text) if sf_1 is not None and sf_1.text else None
        name_parts = [n.text.strip() for n in [df_100.find("subfield[@code='a']"), df_100.find("subfield[@code='j']")] if n is not None and n.text]
        data['source_label'] = " ".join(name_parts) if name_parts else None
    else:
        data['source'], data['source_label'] = None, None
        
    # Target
    df_target = record.find("datafield[@tag='709']") or record.find("datafield[@tag='700']")
    if df_target is not None:
        sf_1 = df_target.find("subfield[@code='1']")
        data['target'] = extract_pim_id(sf_1.text) if sf_1 is not None and sf_1.text else None
        name_parts = [n.text.strip() for n in [df_target.find("subfield[@code='a']"), df_target.find("subfield[@code='j']")] if n is not None and n.text]
        data['target_label'] = " ".join(name_parts) if name_parts else None
    else:
        data['target'], data['target_label'] = None, None

    # WEBES ADATGAZDAGÍTÁS (A javított, precíz kereső)
    if bib_id:
        print(f"[{idx+1}/{len(records)}] Jelzet lekérése a webről: {bib_id}...", end=" ")
        pontos_jelzet = fetch_leltari_szam_from_web(bib_id)
        if pontos_jelzet:
            data['leltari_szam'] = pontos_jelzet
            print(f"Sikeres: {pontos_jelzet}")
        else:
            print("Nem található, maradt az XML alapértelmezett (KLE).")
        time.sleep(1) # Biztonsági szünet a szerver védelmében
        
    extracted_data.append(data)

# Mentés a dúsított JSON fájlba
with open(koztes_json, 'w', encoding='utf-8') as f:
    json.dump(extracted_data, f, ensure_ascii=False, indent=4)

print(f"\nMinden adat elmentve a biztonságos JSON fájlba: {koztes_json}")

In [ ]:
koztes_json = "opac_records.json"
kimeneti_excel = "opac_records.xlsx"

# Beolvasás JSON-ból
with open(koztes_json, 'r', encoding='utf-8') as f:
    mentett_adatok = json.load(f)

# DataFrame építése
df = pd.DataFrame(mentett_adatok)

# Oszlopok sorrendbe rendezése
oszlop_sorrend = [
    'bib.rekord', 'leltari_szam', 'weight', 'date', 
    'megjegyzes', 'source', 'source_label', 'target', 'target_label'
]
df = df.reindex(columns=oszlop_sorrend)

# Mentés Excelbe
df.to_excel(kimeneti_excel, index=False)

print(f"Az Excel táblázat sikeresen legenerálva: {kimeneti_excel}")
df.head()

In [ ]:
df = pd.read_excel("namespace.xlsx", engine="openpyxl")
df = df.where(pd.notnull(df), None)
records = df.to_dict(orient="records")

with open("namespace.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

In [ ]:
# --- FÁJLNEVEK BEÁLLÍTÁSA ---
NAMESPACE_FILE = 'namespace.json'
RECORDS_FILE = 'opac_records.json'
OUTPUT_FILE = 'new_entities.json'

# Fájlok betöltése
with open(NAMESPACE_FILE, 'r', encoding='utf-8') as f:
    namespace = json.load(f)

with open(RECORDS_FILE, 'r', encoding='utf-8') as f:
    records = json.load(f)

print(f"Sikeresen betöltve: {len(namespace)} névtér elem és {len(records)} kapcsolat rekord.")

def extract_pure_name(label_str):
    """Levágja a zárójeles részt a név végéről (pl. 'Váry Rezső (1867-1940)' -> 'Váry Rezső')"""
    if not label_str:
        return ""
    if "(" in label_str:
        return label_str.split("(")[0].strip()
    return label_str.strip()

# 1. Összegyűjtjük a lokális névtérben létező összes ID-t (stringként az összehasonlíthatóság érdekében)
existing_ids = set()
for item in namespace:
    if isinstance(item, dict) and "id" in item:
        if item["id"] is not None:
            existing_ids.add(str(item["id"]).strip())

# 2. Végigmegyünk a kapcsolatokon és kigyűjtjük a hiányzó entitásokat
# Szótárként kell gyűjteni (ID alapján), hogy a duplikációkat automatikusan kiszűrjük
new_entities_dict = {}

for r in records:
    for role in ["source", "target"]:
        id_val = r.get(role)
        label_val = r.get(f"{role}_label")
        
        # Csak akkor nézzük, ha van azonosító és az nem üres vagy "MISSING"
        if id_val is not None and id_val != "MISSING":
            str_id = str(id_val).strip()
            
            # Ha az ID nincs benne a névtérben, és még nem adtuk hozzá a set-hez sem
            if str_id not in existing_ids and str_id not in new_entities_dict:
                pure_name = extract_pure_name(label_val)
                
                # Elkészítjük az új rekordot a kért struktúrával
                new_entities_dict[str_id] = {
                    "Halmaz": None,
                    "id": id_val,  # Megtartja az eredeti típust (szám vagy string)
                    "típus": None,
                    "label": pure_name,
                    "weight": None,
                    "log_suly": None,
                    "névvariáns": None,
                    "dátum": None,
                    "foglalkozás": None,
                    "megjegyzés a foglalkozásról": None
                }

# 3. Lista formátumra alakítás és mentés
new_entities_list = list(new_entities_dict.values())

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(new_entities_list, f, ensure_ascii=False, indent=4)

print("-" * 50)
print(f"Talált új (hiányzó) entitások száma: {len(new_entities_list)}")
print(f"A kimenet elmentve ide: {OUTPUT_FILE}")
print("-" * 50)

In [ ]:
# Opcionális - A new_entities.json átalakítása XLSX formátumra

INPUT_FILE = "new_entities.json"
OUTPUT_FILE = "new_entities.xlsx"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    new_entities = json.load(f)

df_new_entities = pd.DataFrame(new_entities)
df_new_entities.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"Az XLSX fájl elkészült: {OUTPUT_FILE}")
df_new_entities.head()